In [ ]:
# pip install pandas pyarrow

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/Wealthtender Capstone/Modeling/Embeddings/df_embeddings_MVP.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_reviews = pd.read_csv(DATA_PATH)

print("Shape:", df_reviews.shape)
print("Columns:", list(df_reviews.columns))

df_reviews.head(3)

Shape: (4579, 19)
Columns: ['ID', 'Title', 'Date', 'age_years', 'custom_relationship', 'custom_compensation', 'reviewer_name', 'advisor_id', 'advisor_name', 'acf_rating', 'rating', 'review_text_raw', 'total_tokens_raw', 'review_text_processed', 'tokens_processed', 'n_tokens_processed', 'embedding', 'bigrams', 'trigrams']


,ID,Title,Date,age_years,custom_relationship,custom_compensation,reviewer_name,advisor_id,advisor_name,acf_rating,rating,review_text_raw,total_tokens_raw,review_text_processed,tokens_processed,n_tokens_processed,embedding,bigrams,trigrams
0,55476,Absolutely amazing!!!,2026-01-07 15:15:00,0.106776,Current Client,This reviewer received no compensation for thi...,Alyssa Sussman,https://wealthtender.com/financial-advisors/om...,"Omar A. Morillo, CFP®, ChFC®, AIF®",5.0,5.0,Omar Morillo is an exceptional wealth advisor ...,93,is an exceptional wealth advisor and person. h...,"['omar', 'morillo', 'exceptional', 'person.', ...",45,[-2.97033451e-02 4.84776683e-02 -3.40203643e-...,"[('omar', 'morillo'), ('morillo', 'exceptional...","[('omar', 'morillo', 'exceptional'), ('morillo..."
1,55446,It is Simply wonderful working with Omar,2026-01-06 20:44:00,0.109514,Current Client,This reviewer received no compensation for thi...,Nayarit Briceno,https://wealthtender.com/financial-advisors/om...,"Omar A. Morillo, CFP®, ChFC®, AIF®",5.0,5.0,Omar is an exceptional advisor. Beyond the out...,47,is an exceptional advisor. beyond the outstand...,"['omar', 'exceptional', 'advisor.', 'beyond', ...",27,[-5.20202033e-02 1.48930773e-02 -1.24569917e-...,"[('omar', 'exceptional'), ('exceptional', 'adv...","[('omar', 'exceptional', 'advisor.'), ('except..."
2,55445,President,2026-01-06 20:44:00,0.109514,Current Client,This reviewer received no compensation for thi...,Gregory Gutt,https://wealthtender.com/financial-advisors/om...,"Omar A. Morillo, CFP®, ChFC®, AIF®",5.0,5.0,We are extremely happy with the work Omar Mori...,142,we are extremely happy with the work has done ...,"['extremely', 'happy', 'work', 'omar', 'morill...",72,[-3.27313622e-03 4.27362546e-02 -6.80865347e-...,"[('extremely', 'happy'), ('happy', 'work'), ('...","[('extremely', 'happy', 'work'), ('happy', 'wo..."


In [ ]:
def parse_embedding(s):
    # Remove brackets
    s = s.strip()[1:-1]
    # Split by whitespace
    return np.array([float(x) for x in s.split()])

df_reviews["embedding"] = df_reviews["embedding"].apply(parse_embedding)

X = np.vstack(df_reviews["embedding"].values)

X.shape

(4579, 384)

## Baseline "topic modeling": Cluster review embeddings with KMeans

the idea of this is that KMeans groups points into K clusters so that reviews in the same cluster are semantically similar.

Testing multiple values of K, K=20 led to differentiable clusters without overweighing one of them and all clusters have balanced sizes.

In [ ]:
K = 20

kmeans = KMeans(n_clusters=K, random_state=42, n_init="auto")
df_reviews["cluster_id"] = kmeans.fit_predict(X)

# sanity check: cluster sizes
df_reviews["cluster_id"].value_counts().sort_index()

,count
cluster_id,
0,128
1,297
2,219
3,36
4,277
5,247
6,241
7,281
8,221


For each cluster we need to compute cluster centroid and find let's say the 5 reviews closest to that centroid

In [ ]:
centroids = kmeans.cluster_centers_

centroids = kmeans.cluster_centers_

for c in range(20):
    print(f"\n================ CLUSTER {c} ================")

    cluster_idx = df_reviews[df_reviews["cluster_id"] == c].index
    cluster_vectors = X[cluster_idx]

    sims = cosine_similarity(cluster_vectors, centroids[c].reshape(1, -1)).flatten()

    top_idx = cluster_idx[np.argsort(sims)[-5:]]

    for i in top_idx:
        print("\n---")
        print(df_reviews.loc[i, "review_text_processed"][:300])


================ CLUSTER 0 ================

---
we recently started working with and have been extremely pleased so far. they started off by getting to know us holistically - our values, our goals, our priorities - and from there have been helping us to build a comprehensive plan that is tailored to our particular personalities and lifestyles. th

---
and the entire staff have always been helpful and attentive to our financial investments. we've been working with this team for many years. highly recommended!

---
great team that i count on for my retirement planning. i have worked with them for over a year and am very satisfied with products and service based on my investment objectives.

---
the team is very knowledgeable and pleasant to work with! for financial prosperity and growth - cadence wealth partners are the way to go!

---
the team is always very professional and helpful with our financial needs. i trust them to always give us sound advice and to protect our retirement por

Now that we extracted representative reviews per cluster, we have to identify which clusters reflect "relationship value".

For this, we'll extract the most frequent meaningful words inside each cluster.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

for c in sorted(df_reviews["cluster_id"].unique()):

    cluster_texts = df_reviews[df_reviews["cluster_id"] == c]["review_text_processed"]

    vectorizer = TfidfVectorizer(
        max_features=15,
        stop_words="english"
    )

    tfidf = vectorizer.fit_transform(cluster_texts)
    words = vectorizer.get_feature_names_out()

    print(f"\n=== CLUSTER {c} ===")
    print(", ".join(words))


=== CLUSTER 0 ===
family, financial, goals, great, highly, knowledgeable, needs, planning, professional, questions, recommend, team, work, working, years

=== CLUSTER 1 ===
advice, advisor, advisors, family, feel, financial, goals, great, highly, needs, recommend, time, ve, working, years

=== CLUSTER 2 ===
best, experience, financial, goals, great, highly, knowledgeable, needs, professional, recommend, team, ve, work, working, years

=== CLUSTER 3 ===
advisor, clients, family, financial, future, highly, knowledge, knowledgeable, mr, planning, recommend, scipio, services, working, years

=== CLUSTER 4 ===
advice, family, feel, financial, future, goals, guidance, help, plan, planning, retirement, team, time, working, years

=== CLUSTER 5 ===
advice, best, family, feel, financial, goals, great, questions, recommend, retirement, team, time, trust, working, years

=== CLUSTER 6 ===
advice, family, feel, financial, future, goals, help, highly, plan, planner, planning, recommend, team, time

**Comment:** These are VERY homogeneous. So, we'll better split reviews into sentences and we'll embed these sentences, to see if sentence-level clustering separates themes better.

1) Use spaCy for sentence splitting

In [ ]:
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load("en_core_web_sm")

def spacy_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 0]
    return sents

df_reviews["sentences"] = df_reviews["review_text_processed"].apply(spacy_sentences)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
df_sent = (df_reviews[["advisor_id","advisor_name","Date","age_years","rating","sentences", "review_text_processed"]]
           .explode("sentences")
           .rename(columns={"sentences":"sentence"})
           .dropna(subset=["sentence"])
           .reset_index(drop=True))

# filter out super short fragments
df_sent = df_sent[df_sent["sentence"].str.len() >= 40].reset_index(drop=True)

df_sent.tail(10)[["advisor_name","review_text_processed","sentence"]]

,advisor_name,review_text_processed,sentence
19520,Russ Thornton,is not only an incredibly smart and knowledgea...,he was a rock during a difficult time and i hi...
19521,Russ Thornton,"is amazing. if you have the opportunity, work ...","if you have the opportunity, work with him."
19522,Russ Thornton,is a trustworthy financial advisor who special...,is a trustworthy financial advisor who special...
19523,Russ Thornton,is a trustworthy financial advisor who special...,"he is direct, a good communicator and a really..."
19524,Russ Thornton,is a trustworthy financial advisor who special...,he has made it his work to help women who are ...
19525,Russ Thornton,is a trustworthy financial advisor who special...,he is a fiduciary advisor which means that he ...
19526,"Larry Sprung, CFP®",is a true professional. he is not just about s...,he is not just about selling you individual pr...
19527,"Larry Sprung, CFP®",is a true professional. he is not just about s...,he has a big picture plan based on your goals ...
19528,"Larry Sprung, CFP®",is a true professional. he is not just about s...,i have found him to be a great source of infor...
19529,"Larry Sprung, CFP®",is a true professional. he is not just about s...,i highly recommend him to anyone who wants to ...


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
relationship_keywords = [
    "trust", "trustworthy", "honest", "integrity",
    "listen", "listens", "understand", "understands","generous",
    "care", "caring", "compassion", "empathetic",
    "communicate", "communication", "explains", "explain", "clear",
    "patient", "responsive", "available",
    "peace of mind", "secure", "confidence",
    "values", "best interest",
    "life", "divorce", "death", "widow", "retirement", "family",
    "retirement","retire","future"
]

generic_phrases = [
    "highly recommend", "great advisor", "exceptional advisor", "best advisor",
    "knowledgeable", "professional", "years", "team"
]

def keep_relationship_sentence(s):
    s_low = s.lower()
    has_rel = any(k in s_low for k in relationship_keywords)
    too_generic = any(g in s_low for g in generic_phrases)
    # keep if it has relationship signal; allow generic words as long as there's relationship signal
    return has_rel and len(s_low.split()) >= 8

df_sent["keep_rel"] = df_sent["sentence"].apply(keep_relationship_sentence)

df_sent["keep_rel"].mean(), df_sent[df_sent["keep_rel"]].head(10)[["sentence"]]

(np.float64(0.41920122887864825),
                                              sentence
 1   he goes above and beyond in making me feel sec...
 2   because he takes the time to truly listen, he ...
 5   beyond the outstanding customer care provided ...
 8   he is our retirement plan advisor and has also...
 10  he takes the time to clearly explain complex f...
 11  his guidance has given us confidence and peace...
 12  what truly sets apart is his professionalism, ...
 13  we trust his judgment completely and truly val...
 15  always available for any questions or concerns...
 18  is highly intelligent, passionate about his bu...)

Now, we embed only the relationship-salient sentences

In [ ]:
df_rel = df_sent[df_sent["keep_rel"]].copy().reset_index(drop=True)

rel_texts = df_rel["sentence"].tolist()

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

rel_embeddings = model.encode(
    rel_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

rel_embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/128 [00:00<?, ?it/s]

(8187, 384)

Now we will cluster the relationship-salient sentence embeddings with KMeans.

$\ $

We will start with K=12 to avoid over-fragmenting

In [ ]:
K = 12
kmeans_rel = KMeans(n_clusters=K, random_state=42, n_init="auto")
df_rel["topic_id"] = kmeans_rel.fit_predict(rel_embeddings)

df_rel["topic_id"].value_counts().sort_index()

,count
topic_id,
0,355
1,661
2,623
3,574
4,1053
5,714
6,580
7,1148
8,696


In [ ]:
centroids = kmeans_rel.cluster_centers_

for topic in range(K):
    print("\n" + "="*40)
    print(f"TOPIC {topic}")

    # indices for this topic
    idx = df_rel["topic_id"] == topic
    topic_emb = rel_embeddings[idx.values]

    # similarity to centroid
    sims = cosine_similarity(topic_emb, centroids[topic].reshape(1, -1)).flatten()

    # top 5 most representative sentences
    top_idx = np.argsort(sims)[::-1][:5]
    top_sentences = df_rel[idx].iloc[top_idx]["sentence"]

    for s in top_sentences:
        print("-", s)


TOPIC 0
- she is knowledgeable and a pleasure to work with and has a real knack for explaining financial issues that are beyond my limited knowledge.
- she's great at explaining the esoterica intrinsic to financial planning, talking me through my options, making sure i understand, helping me make informed decisions.
- after nearly 30 years knowing and working with , i can say with all sincerity that she delivers sound advice, thoughtful and wise financial planning strategies, as well as timely communication.
- those who have worked with her via my recommendation, all agree that her professionalism, approachability and her ability to focus on their life plans & goals make financial planning smooth and worry-free.
- she is extremely knowledgeable and truly committed to helping people understand and realize their financial potential.

TOPIC 1
- i've worked with for several months and he has been integral in shaping what my financial future looks like and putting me into better financial 

## Ideas for dimensions or "hats":
1. Trust & Integrity
2. Listening & Personalization
3. Communication & Clarity
4. Responsiveness & Availability
5. Life Event Support / Emotional Context
6. Investment & Financial Expertise